# Run pipeline (Raw -> Bronze -> Silver)

Set the parameters in the widgets at the top, then **Run all**.

| Parameter | Values | Meaning |
|---|---|---|
| `load_type` | `full` / `incremental` | Which kind of load this is |
| `source_path` | empty, file, folder, wildcard, or comma separated list | Raw files to load. Empty = `full_load_part*.json` for full, `incremental_full.json` for incremental. Relative paths are inside this repo folder |
| `batch_id` | any text, or `ALL` | Batch identifier. Empty = `full_load` / `incremental_<YYYYMMDD>`. With `layer = silver` it chooses which Bronze batch to process (`ALL` = all batches) |
| `layer` | `all` / `bronze` / `silver` | Which step(s) to run |
| `catalog`, `schema` | | Where the tables are created. Empty catalog = current catalog |

See the README for examples (standard incremental load vs backfill).

In [0]:
%run ./pipeline_utils

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"], "1. Load type")
dbutils.widgets.text("source_path", "", "2. Source path")
dbutils.widgets.text("batch_id", "", "3. Batch ID")
dbutils.widgets.dropdown("layer", "all", ["all", "bronze", "silver"], "4. Layer")
dbutils.widgets.text("catalog", "", "5. Catalog")
dbutils.widgets.text("schema", DEFAULT_SCHEMA, "6. Schema")

params = {name: dbutils.widgets.get(name)
          for name in ["load_type", "source_path", "batch_id", "layer", "catalog", "schema"]}
print(params)

In [0]:
run_id, tables = run_pipeline(**params)

In [0]:
# Log entries written by this run
display(
    spark.table(tables["logs"])
    .filter(F.col("run_id") == run_id)
    .orderBy("start_time")
)

In [0]:
# Current row counts
display(spark.sql(f"""
    SELECT 'bronze' AS table_name, COUNT(*) AS row_count FROM {tables['bronze']}
    UNION ALL SELECT 'silver', COUNT(*) FROM {tables['silver']}
    UNION ALL SELECT 'silver_revisions', COUNT(*) FROM {tables['revisions']}
    UNION ALL SELECT 'quarantine', COUNT(*) FROM {tables['quarantine']}
    UNION ALL SELECT 'execution_logs', COUNT(*) FROM {tables['logs']}
"""))